# Relación juego de videojegos y emociones

La adiccion a videojuegos puede afectar de manera negativa a personas de todas las edades, en este proyecto planeo explorar la informacion de este data set para responder unas preguntas de correlación en cuanto a el tiempo de juego y los problemas existentes o no en cuanto a la salud mental.

preguntas a explorar:  
1. ¿Más horas de juego se asocian con más ansiedad, depresión o estrés?
2. ¿Cómo se relacionan las horas de juego con el sueño y el rendimiento académico o laboral?
3. ¿Difiere el impacto según edad o género?

In [ ]:
# importar librerías
import pandas as pd
import seaborn as sns
from statsmodels.stats.proportion import proportions_ztest
from scipy import stats

In [ ]:
# cargar dataframe

df = pd.read_csv('C:/Users/USUARIO/Documents/TripleTen/Free_proyect/Game_addiction/gaming_addiction.csv')

df.head()

In [ ]:
#exploración inicial.
print('\033[1m'+ '\033[91m'+ 'INFO\n'+ '\033[0m')
print(df.info(),       '\n \n','\033[1m'+ '\033[91m'+ 'NULOS\n'+ '\033[0m')
print(df.isna().sum(), '\n \n','\033[1m'+ '\033[91m'+ 'UNICOS\n'+ '\033[0m')
print(df.nunique(),    '\n')
#no tenemos vacios ni duplicados en Id, genial, limpiemos vacios

for col in df.columns:
    if df[col].dtype in ['float64','int64']: #si son numericos agregemos la media
        df[col] = df[col].fillna(df[col].mean())
    else: #si son texto usemos la moda
        df[col] = df[col].fillna(df[col].mode()[0])

# revicemos si la limpieza fue exitosa

print('\n \n','\033[1m'+ '\033[91m'+ 'NULOS POST LIMPEZA\n'+ '\033[0m')
print(df.isna().sum())


Se limpiaron vacios de la siguente manera
1. 'numericos' se llenan con la media de los datos de la columna.
2. 'texto' se llenan  con la moda de los datos de la columna.

In [ ]:
#miremos si hay outliers con el z score
def numeric_z_analisys(df,columns):
    print('\033[1m'+ '\033[91m'+ 'Z analisys \n'+ '\033[0m')
    for col in columns:
        print('\033[1m'+ '\033[91m'+ col+ '\033[0m')
        mean = df[col].mean()
        std  = df[col].std()
        df['z'] = (df[col] - mean)/std 
        print(df[df['z'].abs() > 3])
        print('\033[1m'+ 'Total over limit: '+ '\033[0m',\
              (df['z'].abs() > 3).sum())
        df.drop(columns=['z'])

interest_cols = ['daily_playtime_hours','stress_score', 'loneliness_score', 'anxiety_level', 'depression_indicator', 'mental_health_risk_score', 'sleep_hours', 'gpa_or_performance_score' ]
numeric_z_analisys(df, interest_cols)

# analisis y visualizaciones.
empecemos por revisar las preguntas planteadas al inicio.
## ¿Más horas de juego se asocian con más ansiedad, depresión o estrés?
datos a usar:  
1. daily_playtime_hours
2. stress_score
3. loneliness_score
4. anxiety_level
5. depression_indicator
6. mental_health_risk_score


In [ ]:
#veamos esas relaciones
hours_vs_emotions_pp = sns.pairplot(df,x_vars=['stress_score', 'loneliness_score', 'anxiety_level', 'depression_indicator', 'mental_health_risk_score'], y_vars=['daily_playtime_hours'])

Las graficas no nos muestran ninguna relación obvia, aunque esto no implica que no haya ninguna, traigamos algunas pruebas estadisticas para verificar

In [ ]:


variables = ['stress_score', 'loneliness_score', 'anxiety_level', 
             'depression_indicator', 'mental_health_risk_score']

for var in variables:
    rho, p = stats.spearmanr(df['daily_playtime_hours'], df[var])
    print(f"{var}: relación={rho:.3f}, p_value={p:.5f}, n_usuarios={df[var].notna().sum()}")

Se encontró una asociación estadísticamente significativa entre las horas de juego diarias y el nivel de estrés (relación=0.303, p<0.001), siendo esta la relación más fuerte identificada.  
También se hallaron asociaciones significativas pero débiles con ansiedad (relación=0.144, p=0.023) y riesgo de salud mental (relación=0.157, p=0.013).  
No se encontró evidencia suficiente de relación entre horas de juego y soledad (p=0.249) ni con el indicador de depresión (p=0.078, marginalmente no significativo). Estos resultados sugieren que, de las variables analizadas, el estrés es la más sensible al tiempo de juego, mientras que la soledad parece estar determinada por otros factores no capturados en esta relación bivariada.

## ¿Cómo se relacionan las horas de juego con el sueño y el rendimiento académico o laboral?
datos a usar:  
1. daily_playtime_hours
2. occupation 
3. sleep_hours
4. gpa_or_performance_score

In [ ]:
hours_vs_productivity_pp = sns.pairplot(df, hue='occupation', y_vars=['daily_playtime_hours'], x_vars=['sleep_hours','gpa_or_performance_score'])

Aqui la visualización nos da un poco mas de pistas, ya que nos muestra primero que encuanto a puntajes de rendimiento, realmente solo tenemos varibilidad en cuanto a los estudiantes. Sin embargo yo encuentro mas interesante que si hay un decline notorio de tiempo de sueño al aumentar las horas de juego, pero ahora veamos si esto es estadisticamente significativo.

In [ ]:
variables_2 = ['sleep_hours','gpa_or_performance_score']

for var in variables_2:
    rho, p = stats.spearmanr(df['daily_playtime_hours'], df[var])
    print(f"{var}: relación={rho:.3f}, p_value={p:.5f}, n_usuarios={df[var].notna().sum()}")

Se encontró una asociación negativa y moderada entre las horas de juego y las horas de sueño (relación=-0.388, p<0.001), la relación más fuerte identificada en el estudio, sugiriendo que un mayor tiempo de juego se vincula con menor descanso.  
En contraste, no se halló evidencia de relación entre las horas de juego y el rendimiento académico/laboral (relación=0.038, p=0.553), lo que indica que, al menos de forma directa, el tiempo de juego no está asociado con el desempeño en esta muestra.

## ¿Difiere el impacto según edad, género o tipo de juego (competitivo, cooperativo, un jugador)?


In [ ]:
def correlaciones_por_grupo(df, columna_grupo, variable_x='daily_playtime_hours'):
    variables_y = ['stress_score', 'loneliness_score', 'anxiety_level', 
                   'depression_indicator', 'mental_health_risk_score',
                   'sleep_hours', 'gpa_or_performance_score']
    
    resultados = []
    for valor_grupo, grupo in df.groupby(columna_grupo):
        for var in variables_y:
            rho, p = stats.spearmanr(grupo[variable_x], grupo[var], nan_policy='omit')
            resultados.append({
                columna_grupo: valor_grupo,
                'variable': var,
                'relación': round(rho, 3),
                'p_value': round(p, 5),
                'significativo': p < 0.05,
                'n': len(grupo)
            })
    return pd.DataFrame(resultados)

df['age_group'] = pd.cut(df['age'], bins=[0, 18, 25, 50], labels=['<18', '18-25', '26-50'])

resultados_genero = correlaciones_por_grupo(df, 'gender')
resultados_edad = correlaciones_por_grupo(df, 'age_group')

In [ ]:
resultados_genero

Al desagregar por género, se observa que la asociación entre horas de juego y estrés, depresión, y riesgo de salud mental es estadísticamente significativa únicamente en el grupo masculino (n=158), mientras que en el grupo femenino (n=66) estas relaciones no alcanzan significancia.  
En contraste, la relación entre horas de juego y horas de sueño es significativa y de magnitud moderada-fuerte en ambos géneros, siendo más pronunciada en mujeres (relación=-0.534) que en hombres (relación=-0.329).  
Los grupos ‘No binarios’ (n=18) y ‘Prefiere no decir’ (n=8) tuvieron un tamaño de muestra insuficiente para un análisis estadístico confiable, por lo que sus resultados no se interpretan como concluyentes.

In [ ]:
resultados_edad

El impacto de las horas de juego varía notablemente según la edad. El grupo de 26-50 años mostró asociaciones significativas con todas las variables de salud mental evaluadas (estrés relación=0.466, ansiedad relación=0.324, depresión relación=0.232, riesgo de salud mental relación=0.317; todas p<0.05), siendo el grupo más consistentemente afectado. Curiosamente, en este mismo grupo la soledad mostró una asociación negativa (relación=-0.273), sugiriendo que el juego podría cumplir una función social que reduce el aislamiento en adultos, contrario al patrón esperado.  
En los grupos <18 y 18-25, solo el estrés y las horas de sueño mostraron relaciones significativas. La relación con el sueño fue la más consistente across todos los grupos de edad (relación entre -0.355 y -0.419, p<0.001 en los tres), confirmándose como el hallazgo más robusto del estudio.